In [ ]:
import pandas as pd
import numpy as np
import os
import glob

In [ ]:
#Merging csv files

path=os.getcwd()
files=glob.glob(os.path.join(path,'*.csv'))

column_names=['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR','HTHG','HTAG','HTR','HY','AY','HR','AR']
datalist=[]

for f in files:
    data=pd.read_csv(f,header=0,usecols=column_names)
    data['season']=os.path.basename(f).split('.')[0]
    datalist.append(data)
df=pd.concat(datalist,ignore_index=True)

In [ ]:
df.info()

In [ ]:
sample = pd.DataFrame({'HomeTeam': {0: 'A', 1: 'B', 2: 'A'},
                       'AwayTeam': {0: 'B', 1: 'C', 2: 'C'},
                       'Value1':   {0: 1,   1: 3,   2: 5},
                       'Value2':   {0: 2,   1: 4,   2: 6},
                      })
print(sample)
print('----------')
print('After melting :')
print('----------')
print(sample.melt(id_vars=['Value1','Value2'],value_vars=['HomeTeam','AwayTeam']))

In [ ]:
df.head()

In [ ]:
# Copying (melting) records to have 'Away Team' column result in rows too

df['Home']=df['HomeTeam']
df['Away']=df['AwayTeam']
df=df.melt(id_vars=['Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR','HTHG','HTAG','HTR','HY','AY','HR','AR','season'],
        value_vars=['Home','Away'],
        var_name='HomeAway',
        value_name='Team')

@np.vectorize
def result(team,home_team,home_goal,away_goal):
    if team==home_team:
        if home_goal>away_goal:
            return 3
        elif home_goal<away_goal:
            return 0
        else:
            return 1
    else:
         if home_goal>away_goal:
            return 0
         elif home_goal<away_goal:
            return 3
         else:
            return 1       

df['MatchResult']=result(df['Team'],df['HomeTeam'],df['FTHG'],df['FTAG'])
df['home_goal']=np.where(df['Team']==df['HomeTeam'],df['FTHG'],df['FTAG'])
df['away_goal']=np.where(df['Team']!=df['HomeTeam'],df['FTHG'],df['FTAG'])
print(df.shape)
df.sort_values(['Date']).head(20)

In [ ]:
# Calculating the league table items Season 2018/19

league=df.loc[df['season']=='20182019']
# league.groupby(['Team'])['MatchResult'].agg('sum')

def table_league(frame):
    
    return pd.DataFrame({'Played':np.size(frame['Team']),
                         'Won':np.sum(frame['MatchResult']==3),
                         'Drawn':np.sum(frame['MatchResult']==1),
                         'Lost':np.sum(frame['MatchResult']==0),
                         'GF':np.sum(frame['home_goal']),
                         'GA':np.sum(frame['away_goal']),
                         'GD':np.sum(frame['home_goal'])-np.sum(frame['away_goal']),
                         'Points':np.sum(frame['MatchResult'])
                        },index=frame['Team'].unique())

                            
cal_leauge=league.groupby(['Team'])
result_table=cal_leauge.apply(table_league)
result_table.sort_values(['Points'],ascending=False)